## 1. Data Preprocessing & Cleaning

### 1.1. Import Required Libraries

In [52]:
import pandas as pd
import numpy as np
import os
from sklearn.impute import SimpleImputer

os.chdir("C:\Projects\Predicting Football Player Values\Data")

<>:6: SyntaxWarning: invalid escape sequence '\P'
<>:6: SyntaxWarning: invalid escape sequence '\P'
C:\Users\azedd\AppData\Local\Temp\ipykernel_18904\437665199.py:6: SyntaxWarning: invalid escape sequence '\P'
  os.chdir("C:\Projects\Predicting Football Player Values\Data")


### 1.2. Load dataset 

In [53]:
gk_df = pd.read_csv('GK-FBREF.csv')
Market_Value_df = pd.read_csv('PlayersTM.csv')

### 1.3. Clean data

In [54]:
def add_market_value_to_gk_df(gk_df, Market_Value_df):
    # Clean market values data
    def clean_market_value(value):
        if pd.isna(value) or value == '-':
            return np.nan
        value = str(value).replace('€', '').replace(' ', '')
        if 'mio' in value:
            value = value.replace('mio.', '').replace(',', '.')
            return float(value) * 1_000_000
        elif 'K' in value:
            value = value.replace('K', '')
            return float(value) * 1_000
        else:
            return np.nan

    Market_Value_df['Market_Value_Numeric'] = Market_Value_df['Market_Value'].apply(clean_market_value)
    merged_df = gk_df.merge(
        Market_Value_df[['Player_Name', 'Market_Value_Numeric']],
        left_on='Player', right_on='Player_Name', how='left'
    )
    merged_df.drop('Player_Name', axis=1, inplace=True)
    return merged_df

df = add_market_value_to_gk_df(gk_df, Market_Value_df)


In [55]:
def preprocess_gk_data(df):
    # Convert comma-separated numbers
    for col in ['Min']:
        df[col] = df[col].astype(str).str.replace(',', '').astype(float)

    # Convert percentage strings to numeric
    for col in ['Save%', 'CS%', 'PKSave%']:
        df[col] = pd.to_numeric(df[col].astype(str).str.replace('%', ''), errors='coerce') / 100

    # Create basic features
    df['Mins_per_match'] = df['Min'] / df['MP']
    df['Shots_faced_per_90'] = df['SoTA'] / df['90s']
    df['Saves_per_90'] = df['Saves'] / df['90s']
    df['xGA_per_shot'] = df['GA'] / df['SoTA']  # Expected goals against per shot

    # Handle missing values
    for col in ['Save%', 'PKSave%']:
        df[col].fillna(df[col].median(), inplace=True)

    return df

gk_clean = preprocess_gk_data(df)


C:\Users\azedd\AppData\Local\Temp\ipykernel_18904\3218533721.py:18: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].median(), inplace=True)
C:\Users\azedd\AppData\Local\Temp\ipykernel_18904\3218533721.py:18: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For exa

In [56]:
gk_clean

,Player,Nation,Pos,Team,Age,Born,MP,Starts,Min,90s,...,PKatt,PKA,PKsv,PKm,PKSave%,Market_Value_Numeric,Mins_per_match,Shots_faced_per_90,Saves_per_90,xGA_per_shot
0,Alisson,BRA,GK,Liverpool,31,1992,28,28,2508.0,27.9,...,1,1,0,0,0.000,20000000.0,89.571429,3.584229,2.616487,0.290000
1,Alphonse Areola,FRA,GK,West Ham,31,1993,26,25,2260.0,25.1,...,0,0,0,0,0.000,9000000.0,86.923077,4.581673,3.067729,0.356522
2,Kepa Arrizabalaga,ESP,GK,Bournemouth,29,1994,31,31,2790.0,31.0,...,4,4,0,0,0.000,10000000.0,90.000000,4.322581,3.064516,0.291045
3,Brandon Austin,ENG,GK,Tottenham,25,1999,1,1,90.0,1.0,...,0,0,0,0,0.000,600000.0,90.000000,4.000000,2.000000,0.500000
4,Altay Bayındır,TUR,GK,Manchester Utd,26,1998,4,4,360.0,4.0,...,0,0,0,0,0.000,8000000.0,90.000000,4.250000,2.000000,0.588235
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
201,Matvei Safonov,RUS,GK,Paris S-G,25,1999,10,9,879.0,9.8,...,2,2,0,0,0.000,NaN,87.900000,2.551020,1.734694,0.360000
202,Brice Samba,FRA,GK,Rennes,30,1994,17,17,1529.0,17.0,...,3,1,2,0,0.667,12000000.0,89.941176,5.117647,3.411765,0.310345
203,Brice Samba,FRA,GK,Lens,30,1994,15,15,1350.0,15.0,...,4,4,0,0,0.000,12000000.0,90.000000,3.600000,2.666667,0.259259
204,Arnau Tenas,ESP,GK,Paris S-G,23,2001,1,1,90.0,1.0,...,0,0,0,0,0.000,3000000.0,90.000000,4.000000,3.000000,0.250000


## 2. Feature Engineering

In [57]:
# Feature engineering functions
def create_performance_features(df):
    """Create advanced performance metrics"""
    # Shot-stopping metrics
    df['Goals_prevented'] = (df['Save%'] - df['Save%'].median()) * df['SoTA']
    df['xSave_performance'] = (df['Save%'] - df['Save%'].quantile(0.25)) / (df['Save%'].quantile(0.75) - df['Save%'].quantile(0.25))
    
    # Penalty performance
    df['Penalty_save_rate'] = df['PKSave%'] / 100
    df['Penalty_save_rate'].fillna(0, inplace=True)
    df['Penalty_metrics'] = np.where(
        df['PKatt'] > 0,
        df['Penalty_save_rate'] * np.log1p(df['PKatt']),
        0
    )
    
    # Consistency metrics
    df['Save_consistency'] = 1 / (1 + (df['Save%'] - df['Save%'].mean())**2)
    df['Min_consistency'] = df['Min'] / (df['90s'] * 90 * 1.05)  # Account for injury time
    
    # Clean sheet metrics
    df['CS_per_match'] = df['CS'] / df['Starts']
    df['CS_efficiency'] = df['CS'] / (df['GA'] + 1e-5)  # Avoid division by zero
    
    return df

In [58]:
def create_contextual_features(df):
    """Add contextual features using external data"""
    # League strength mapping (real UEFA coefficients)
    league_strength = {
        'Premier League': 0.90839, 'La Liga': 0.74953, 'Bundesliga': 0.71117,
        'Serie A': 0.80946, 'Ligue 1': 0.65177
    }
    
    # Map teams to leagues (simplified - would be automated in production)
    df['League'] = df['Team'].apply(lambda x: 
        'Premier League' if x in ['Liverpool', 'Manchester City', 'Arsenal', 'Manchester Utd', 'Chelsea', 'Tottenham', 'Leicester City', 'West Ham', 'Aston Villa', 'Newcastle Utd', 'Brighton', 'Brentford', 'Crystal Palace', 'Fulham', 'Wolves', 'Bournemouth', "Nott'ham Forest", 'Everton', 'Southampton','Ipswich Town'] else
        'La Liga' if x in ['Real Madrid', 'Barcelona', 'Atlético Madrid', 'Sevilla', 'Real Sociedad', 'Betis', 'Valencia', 'Athletic Club', 'Villarreal', 'Celta Vigo', 'Getafe', 'Osasuna', 'Espanyol', 'Alavés', 'Mallorca', 'Leganés', 'Rayo Vallecano', 'Valencia', 'Girona', 'Las Palmas'] else
        'Bundesliga' if x in ['Bayern Munich', 'Borussia Dortmund', 'RB Leipzig', 'Bayer Leverkusen', 'Borussia Monchengladbach', 'VfL Wolfsburg', 'Eintracht Frankfurt', 'SC Freiburg', 'Hoffenheim', 'Hertha Berlin', 'Mainz 05', 'FC Augsburg', 'VfB Stuttgart', 'Union Berlin', 'Schalke 04', 'Cologne', 'Werder Bremen', 'Arminia Bielefeld'] else
        'Serie A' if x in ['Juventus', 'AC Milan', 'Inter Milan', 'Napoli', 'AS Roma', 'Lazio', 'Atalanta', 'Fiorentina', 'Sassuolo', 'Torino', 'Bologna', 'Cagliari', 'Sampdoria', 'Genoa', 'Udinese', 'Empoli', 'Hellas Verona'] else
        'Ligue 1' if x in ['Paris Saint-Germain', 'Lille OSC', 'Monaco', 'Olympique Lyonnais', 'Marseille', 'Rennes', 'Nice', 'Montpellier', 'Strasbourg', 'Brest', 'Nantes', 'Reims', 'Angers SCO', 'Toulouse FC', 'Lorient', 'Clermont Foot', 'Auxerre', 'Ajaccio'] else
        'Other'
    )
    
    df['League_strength'] = df['League'].map(league_strength)
    

    # Adjusted performance metrics
    df['Adj_save%'] = df['Save%'] * df['League_strength']
    df['Adj_GA90'] = df['GA90'] / df['League_strength']
    
    # Big match performance proxy
    top_teams = ['Liverpool', 'Manchester City', 'Real Madrid', 'Bayern Munich', 'Barcelona', 'Paris Saint-Germain', 'Juventus', 'AC Milan', 'Inter Milan', 'Atletico Madrid', 'Chelsea', 'Arsenal', 'Tottenham', 'Borussia Dortmund']
    df['Top_team_experience'] = df['Team'].isin(top_teams).astype(int)
    
    return df


In [59]:
def create_composite_features(df):
    """Create composite performance indices"""
    # Shot-stopping index
    df['Shot_stopping_index'] = (
        0.4 * (df['Save%'] / df['Save%'].max()) + 
        0.3 * (df['Goals_prevented'] / df['Goals_prevented'].max()) +
        0.3 * (1 - df['GA90'] / df['GA90'].max())
    )
    # Command and control index
    df['Command_index'] = (
        0.5 * (df['CS_per_match'] / df['CS_per_match'].max()) +
        0.3 * (df['Min_consistency'] / df['Min_consistency'].max()) +
        0.2 * (df['Penalty_metrics'] / (df['Penalty_metrics'].max() + 1e-5))
    )
    # Age-based performance curve (peak age 28-32)
    df['Age_factor'] = np.where(
        df['Age'] < 25, 0.7 + (df['Age'] - 18) * 0.05,
        np.where(df['Age'] > 32, 1.0 - (df['Age'] - 32) * 0.03,
        1.0)
    )
    # Experience metric
    df['Experience'] = np.log1p(df['Min']) * df['Age_factor']
    
    # Market value drivers
    df['Market_value_driver'] = (
        0.35 * df['Shot_stopping_index'] +
        0.25 * df['Command_index'] +
        0.20 * df['Experience'] +
        0.10 * df['League_strength'] +
        0.10 * df['Top_team_experience']
    )
    
    return df

In [60]:
def feature_selection(df, target_col='Market_value_driver'):
    """Select final features using correlation analysis and mandatory inclusion"""
    # Calculate correlations
    corr_matrix = df.corr(numeric_only=True)
    target_corr = corr_matrix[target_col].sort_values(ascending=False)
    
    # Select features with |correlation| > 0.15
    selected_features = target_corr[abs(target_corr) > 0.15].index.tolist()
    
    # Mandatory features
    mandatory_features = [
        'Shot_stopping_index', 'Command_index', 'Experience',
        'League_strength', 'Age', 'Market_value_driver'
    ]
    for feature in mandatory_features:
        if feature not in selected_features and feature in df.columns:
            selected_features.append(feature)
    
    # Return dataframe with selected features
    return df[list(set(selected_features))]


In [61]:

# Main pipeline
def feature_engineering_pipeline(df):

    # Step 1: Create performance features
    df = create_performance_features(df)
    
    # Step 2: Add contextual features
    df = create_contextual_features(df)
    
    # Step 3: Create composite features
    df = create_composite_features(df)
 
    # Step 4: Feature selection
    final_df = feature_selection(df)
    
    # Save engineered dataset
    final_df.to_csv('engineered_gk_features.csv', index=False)
    
    print("Feature engineering complete!")
    print(f"Original features: {len(gk_df.columns)}")
    print(f"Engineered features: {len(final_df.columns)}")
    print(f"Total features created: {len(final_df.columns) - len(gk_df.columns)}")
    
    return final_df

# Execute pipeline
if __name__ == "__main__":
    engineered_data = feature_engineering_pipeline(gk_clean)

Feature engineering complete!
Original features: 25
Engineered features: 37
Total features created: 12


C:\Users\azedd\AppData\Local\Temp\ipykernel_18904\2314859639.py:10: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Penalty_save_rate'].fillna(0, inplace=True)
